# 02 · Comprensión de los datos de BCS (fotos y bitácora)

Fase 2 de CRISP-DM para los datos que tienen BCS: las fotos de la visita al CAETEC del 14/10/2026, la bitácora de los expertos (`etiquetas_bsc.xlsx`) y el dataset `vms-3.csv`. Los CSV de ordeño solo se usan para cruzar los IDs de las vacas.

Los datos no están en el repositorio. Su ubicación se configura en la celda siguiente (`RAIZ`).

In [ ]:
from pathlib import Path
from collections import Counter
import datetime as dt
import hashlib
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from PIL import Image

try:
    import pillow_heif
    pillow_heif.register_heif_opener()
    LEE_HEIC = True
except ImportError:
    LEE_HEIC = False

RAIZ = Path.home() / "Downloads" / "retoia"
DIR_FOTOS = RAIZ / "fotosextraidas"
BITACORA = RAIZ / "etiquetas_bsc.xlsx"
DIR_CSV = RAIZ / "csv"
VMS3 = DIR_CSV / "vms-3.csv"

# Las gráficas se guardan junto al notebook
FIGURAS = Path("figuras")
FIGURAS.mkdir(exist_ok=True)

pd.set_option("display.max_rows", 120)
for p in (DIR_FOTOS, BITACORA, DIR_CSV, VMS3):
    print("OK       " if p.exists() else "NO EXISTE", p)
print("Lectura de HEIC:", "sí" if LEE_HEIC else "no (instala pillow-heif)")

In [ ]:
def sha256(ruta, bloque=1 << 20):
    """Huella del contenido del archivo: dos archivos con la misma huella son idénticos."""
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(bloque), b""):
            h.update(b)
    return h.hexdigest()


def metadatos(ruta):
    """Tamaño, cámara, fecha, orientación y profundidad de una foto (sin decodificar los píxeles)."""
    r = {"ancho": None, "alto": None, "camara": "", "fecha": "", "orientacion": None,
         "profundidad": False, "error": ""}
    try:
        with Image.open(ruta) as im:
            exif = im.getexif()
            r["ancho"], r["alto"] = im.size
            r["camara"] = f"{exif.get(0x010F, '')} {exif.get(0x0110, '')}".strip()
            fecha = exif.get_ifd(0x8769).get(0x9003) or exif.get(0x0132) or ""
            r["fecha"] = str(fecha)[:10].replace(":", "-")
            r["orientacion"] = exif.get(0x0112)
            r["profundidad"] = bool(im.info.get("depth_images"))
    except Exception as e:
        r["error"] = str(e)[:120]
    return r


def norm(texto):
    """Nombre de columna en minúsculas, sin acentos ni símbolos ('ID - VACA' -> 'idvaca')."""
    t = str(texto).lower().translate(str.maketrans("áéíóúüñ", "aeiouun"))
    return re.sub(r"[^a-z0-9]", "", t)


def texto(v):
    """Celda de Excel a texto (1221.0 -> '1221', una hora -> 'HH MM')."""
    if v is None:
        return ""
    if isinstance(v, float) and v.is_integer():
        return str(int(v))
    if isinstance(v, (dt.time, dt.datetime)):
        return v.strftime("%H %M")
    return str(v).strip()


def clave(x):
    """Ordena los IDs como números cuando lo son."""
    return (0, int(x)) if str(x).isdigit() else (1, str(x))


def ordenar_ids(ids):
    return sorted(ids, key=clave)

## 2.1 Recolección de los datos

Inventario de las fuentes: cuántos archivos hay y cuánto pesan. Para cada foto se calcula su huella (SHA-256) y se leen sus metadatos.

In [ ]:
EXT_FOTO = {".heic", ".heif", ".jpg", ".jpeg", ".png"}
filas = []
for p in sorted(DIR_FOTOS.rglob("*")):
    if p.is_file() and p.suffix.lower() in EXT_FOTO and not p.name.startswith("."):
        rel = p.relative_to(DIR_FOTOS)
        filas.append({"id_vaca": p.parent.name.strip(), "descarga": rel.parts[0],
                      "archivo": p.name, "ruta": str(rel), "ext": p.suffix.lower(),
                      "bytes": p.stat().st_size, "sha256": sha256(p)})
fotos = pd.DataFrame(filas)
fotos = pd.concat([fotos, pd.DataFrame([metadatos(DIR_FOTOS / r) for r in fotos["ruta"]])], axis=1)

csv_ordeno = sorted(p for p in DIR_CSV.rglob("*.csv") if p.stem.strip().isdigit())
ids_ordeno = {p.stem.strip() for p in csv_ordeno}

fuentes = pd.DataFrame([
    {"fuente": "Fotos (fotosextraidas/)", "archivos": len(fotos), "MB": fotos["bytes"].sum() / 1e6},
    {"fuente": "Bitácora de expertos (etiquetas_bsc.xlsx)", "archivos": 1, "MB": BITACORA.stat().st_size / 1e6},
    {"fuente": "Dataset VMS 3 (vms-3.csv)", "archivos": 1, "MB": VMS3.stat().st_size / 1e6},
    {"fuente": "CSV de ordeño (uno por vaca)", "archivos": len(csv_ordeno),
     "MB": sum(p.stat().st_size for p in csv_ordeno) / 1e6},
]).round({"MB": 2})
display(fuentes)

In [ ]:
# Bitácora: se busca la fila que tiene los encabezados "ID" y "BCS" (puede no ser la primera)
wb = load_workbook(BITACORA, read_only=True, data_only=True)
hoja, fila_enc = None, None
for ws in wb.worksheets:
    for i, fila in enumerate(ws.iter_rows(min_row=1, max_row=40, values_only=True), start=1):
        nombres = [norm(v) for v in fila if v not in (None, "")]
        if any(n.startswith("id") for n in nombres) and any(n.startswith("bcs") for n in nombres):
            hoja, fila_enc = ws, i
            break
    if hoja is not None:
        break
nombre_hoja = hoja.title
filas = list(hoja.iter_rows(min_row=fila_enc, values_only=True))
wb.close()

encabezado = [texto(c) or f"(columna {i + 1})" for i, c in enumerate(filas[0])]
datos = []
for num, fila in enumerate(filas[1:], start=fila_enc + 1):
    valores = [texto(c) for c in fila][:len(encabezado)]
    if any(valores):
        datos.append([num] + valores + [""] * (len(encabezado) - len(valores)))
bit_raw = pd.DataFrame(datos, columns=["fila_excel"] + encabezado)


def columna(df, *prefijos):
    for c in df.columns:
        if c != "fila_excel" and any(norm(c).startswith(p) for p in prefijos):
            return c
    raise KeyError(f"No encontré una columna que empiece con {prefijos}")


c_id, c_hora, c_bcs = columna(bit_raw, "idvaca", "id"), columna(bit_raw, "hora"), columna(bit_raw, "bcs")
bit = pd.DataFrame({
    "fila_excel": bit_raw["fila_excel"],
    "id_vaca": bit_raw[c_id].str.replace(r"\.0$", "", regex=True),
    "hora_original": bit_raw[c_hora],
    "bcs": pd.to_numeric(bit_raw[c_bcs].str.replace(",", "."), errors="coerce"),
})
print(f"Bitácora: hoja '{nombre_hoja}', encabezados en la fila {fila_enc}, {len(bit)} filas")
print("Columnas:", [c for c in bit_raw.columns if c != "fila_excel"])

In [ ]:
# Dataset VMS 3: se prueba la codificación y se detecta el separador
def leer_csv(ruta):
    for enc in ("utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(ruta, sep=None, engine="python", encoding=enc,
                               dtype=str, keep_default_na=False)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"No pude leer {ruta}")


vms = leer_csv(VMS3)
vms.columns = [str(c).strip() for c in vms.columns]
columnas_vms = list(vms.columns)
c_vms_id = next((c for c in columnas_vms if norm(c).startswith("numerodelanimal")), columnas_vms[0])
c_evento = next((c for c in columnas_vms if "eventobcs" in norm(c)), None)
vms["id_vaca"] = vms[c_vms_id].str.strip().str.replace(r"\.0$", "", regex=True)
print(f"vms-3.csv: {len(vms)} filas y {len(columnas_vms)} columnas")
print(f"ID de la vaca: '{c_vms_id}' | Evento BCS: '{c_evento}'")
print(f"CSV de ordeño: {len(csv_ordeno)} vacas")

## 2.2 Descripción de los datos

### Bitácora de expertos

In [ ]:
hora = bit["hora_original"].str.strip()
formato_hora = pd.Series(np.select(
    [hora.str.lower().eq("x"), hora.str.match(r"^\d{1,2}\s\d{2}$")],
    ["x (sin dato)", "HH MM"], "otro formato"), index=bit.index)

resumen_bit = pd.Series({
    "evaluaciones (filas)": len(bit),
    "vacas distintas": bit["id_vaca"].nunique(),
    "vacas con más de una evaluación": int((bit["id_vaca"].value_counts() > 1).sum()),
    "BCS mínimo": bit["bcs"].min(),
    "BCS máximo": bit["bcs"].max(),
    "BCS promedio": round(bit["bcs"].mean(), 3),
    "BCS mediana": bit["bcs"].median(),
    "BCS desviación estándar": round(bit["bcs"].std(), 3),
    "todos en pasos de 0.25": bool(((bit["bcs"].dropna() * 4) % 1 == 0).all()),
}, dtype=object)
display(resumen_bit.to_frame("valor"))
display(formato_hora.value_counts().rename_axis("formato de la hora").to_frame("filas"))

### Fotografías

Las cifras "únicas" cuentan una sola vez los archivos con el mismo contenido (misma huella), aunque estén repetidos o tengan otro nombre.

In [ ]:
unicas = fotos.drop_duplicates("sha256")
resumen_fotos = pd.Series({
    "archivos": len(fotos),
    "fotos únicas": len(unicas),
    "tamaño total (GB)": round(fotos["bytes"].sum() / 1e9, 2),
    "tamaño sin copias (GB)": round(unicas["bytes"].sum() / 1e9, 2),
    "carpetas descargadas": fotos[["id_vaca", "descarga"]].drop_duplicates().shape[0],
    "vacas (IDs de carpeta)": fotos["id_vaca"].nunique(),
    "fotos con mapa de profundidad": int(unicas["profundidad"].sum()),
    "fotos que no se pudieron leer": int((unicas["error"] != "").sum()),
}, dtype=object)
display(resumen_fotos.to_frame("valor"))

display(unicas["ext"].value_counts().rename_axis("formato").to_frame("fotos únicas"))
display(unicas["camara"].replace("", "(sin dato)").value_counts().rename_axis("cámara").to_frame("fotos únicas"))
resolucion = unicas["ancho"].astype("Int64").astype(str) + " × " + unicas["alto"].astype("Int64").astype(str)
display(resolucion.value_counts().rename_axis("resolución (px)").to_frame("fotos únicas"))
display(unicas["fecha"].replace("", "(sin dato)").value_counts().rename_axis("fecha de captura").to_frame("fotos únicas"))

In [ ]:
por_vaca = fotos.drop_duplicates(["id_vaca", "sha256"]).groupby("id_vaca").size().rename("fotos únicas")
por_vaca = por_vaca.loc[ordenar_ids(por_vaca.index)]
display(por_vaca.describe().round(2).to_frame("fotos únicas por vaca"))
display(por_vaca.to_frame())

## 2.3 Exploración de los datos

### Distribución del BCS

In [ ]:
bcs_validos = bit["bcs"].dropna()
clases = np.arange(np.floor(bcs_validos.min() * 4) / 4, bcs_validos.max() + 0.125, 0.25)
conteo = bcs_validos.value_counts().reindex(clases, fill_value=0)
tabla_clases = pd.DataFrame({
    "evaluaciones": conteo.values,
    "porcentaje": (100 * conteo / conteo.sum()).round(1).values,
}, index=pd.Index([f"{c:.2f}" for c in clases], name="BCS"))
display(tabla_clases)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(tabla_clases.index, tabla_clases["evaluaciones"], color="#4C72B0")
for x, y in zip(tabla_clases.index, tabla_clases["evaluaciones"]):
    ax.text(x, y + 0.3, str(y), ha="center", fontsize=9)
ax.set_xlabel("BCS asignado por los expertos")
ax.set_ylabel("Número de evaluaciones")
ax.set_title(f"Distribución del BCS ({len(bcs_validos)} evaluaciones de {bit['id_vaca'].nunique()} vacas)")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURAS / "distribucion_bcs.png", dpi=200)
plt.show()

### Vacas con más de una evaluación

A algunas vacas las evaluaron dos veces (dos expertos, o la misma vaca en otro momento). La diferencia entre evaluaciones sirve como referencia de cuánto varía el BCS humano.

In [ ]:
rep = bit[bit.duplicated("id_vaca", keep=False)].sort_values(["id_vaca", "fila_excel"])
rep = rep.assign(orden=rep.groupby("id_vaca").cumcount() + 1)
pares = rep.pivot(index="id_vaca", columns="orden", values="bcs")
pares.columns = [f"bcs_{c}" for c in pares.columns]
pares["diferencia"] = pares.max(axis=1) - pares.min(axis=1)
pares = pares.loc[ordenar_ids(pares.index)]
display(pares)
dif = pares["diferencia"]
print(f"{len(pares)} vacas con más de una evaluación | diferencia media {dif.mean():.3f}, "
      f"máxima {dif.max():.2f}, idénticas {(dif == 0).sum()}")

### Fotos por vaca

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(por_vaca, bins=range(1, por_vaca.max() + 2), align="left", rwidth=0.85, color="#4C72B0")
ax.set_xticks(range(1, por_vaca.max() + 1))
ax.set_xlabel("Fotos únicas por vaca")
ax.set_ylabel("Número de vacas")
ax.set_title(f"Fotos por vaca ({len(por_vaca)} vacas)")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURAS / "fotos_por_vaca.png", dpi=200)
plt.show()

### Cruce entre fuentes

Qué vacas aparecen en cada fuente: bitácora de expertos, carpetas de fotos, `vms-3.csv` y CSV de ordeño.

In [ ]:
ids_bit, ids_fotos, ids_vms = set(bit["id_vaca"]), set(fotos["id_vaca"]), set(vms["id_vaca"])
conteos = pd.Series({
    "en la bitácora": len(ids_bit),
    "bitácora y fotos": len(ids_bit & ids_fotos),
    "bitácora y vms-3": len(ids_bit & ids_vms),
    "bitácora y ordeño": len(ids_bit & ids_ordeno),
    "bitácora, vms-3 y ordeño": len(ids_bit & ids_vms & ids_ordeno),
    "en vms-3 sin BCS de expertos": len(ids_vms - ids_bit),
})
display(conteos.to_frame("vacas"))
print("Bitácora y ordeño:", ordenar_ids(ids_bit & ids_ordeno))
print("Bitácora, vms-3 y ordeño:", ordenar_ids(ids_bit & ids_vms & ids_ordeno))
print("En vms-3 sin BCS de expertos:", ordenar_ids(ids_vms - ids_bit))

cruce = pd.DataFrame({"id_vaca": ordenar_ids(ids_bit | ids_fotos | ids_vms)})
for nombre, ids in (("bitácora", ids_bit), ("fotos", ids_fotos), ("vms-3", ids_vms), ("ordeño", ids_ordeno)):
    cruce[nombre] = np.where(cruce["id_vaca"].isin(ids), "Sí", "No")
display(cruce)

### "Evento BCS" del dataset contra el BCS de los expertos

Se compara el promedio de las evaluaciones de los expertos de cada vaca con la columna "Evento BCS" de `vms-3.csv`.

In [ ]:
if c_evento is None:
    print("vms-3.csv no tiene una columna 'Evento BCS'")
else:
    evento = vms[["id_vaca", c_evento]].rename(columns={c_evento: "evento_bcs"})
    evento["evento_bcs"] = pd.to_numeric(evento["evento_bcs"].str.replace(",", "."), errors="coerce")
    expertos = bit.groupby("id_vaca")["bcs"].agg(bcs_expertos="mean", evaluaciones="size").reset_index()
    comp = expertos.merge(evento.dropna(subset=["evento_bcs"]), on="id_vaca")
    comp["diferencia"] = (comp["evento_bcs"] - comp["bcs_expertos"]).round(2)
    display(comp.sort_values("diferencia", ascending=False).reset_index(drop=True))
    print(f"{len(comp)} vacas con ambos valores: Evento BCS mayor en {(comp['diferencia'] > 0).sum()}, "
          f"igual en {(comp['diferencia'] == 0).sum()} y menor en {(comp['diferencia'] < 0).sum()}")

## 2.4 Verificación de la calidad de los datos

In [ ]:
# Copias exactas y descargas repetidas
copias = len(fotos) - len(fotos.drop_duplicates(["id_vaca", "sha256"]))
numeradas = int(fotos["archivo"].str.contains(r"\(\d+\)\.", regex=True).sum())
descargas = fotos.groupby(["id_vaca", "descarga"])["sha256"].apply(frozenset)
dobles = {v: ("mismo contenido" if len(set(g.values)) == 1 else "contenido distinto")
          for v, g in descargas.groupby(level=0) if len(g) > 1}
print(f"Archivos repetidos dentro de una misma vaca: {copias}")
print(f"Archivos con nombre tipo '(1)': {numeradas}")
print(f"Vacas con más de una carpeta descargada: {len(dobles)}")
display(pd.Series(dobles, dtype=object).rename_axis("id_vaca").to_frame("carpetas"))

In [ ]:
# Fotos idénticas asignadas a más de una vaca
vacas_por_foto = (fotos.drop_duplicates(["id_vaca", "sha256"])
                  .groupby("sha256")["id_vaca"].apply(lambda s: ordenar_ids(set(s))))
compartidas = vacas_por_foto[vacas_por_foto.map(len) > 1]
grupos_compartidos = Counter(", ".join(v) for v in compartidas)
print(f"Fotos idénticas en más de una vaca: {len(compartidas)}")
display(pd.Series(grupos_compartidos, dtype=object).rename_axis("vacas").to_frame("fotos"))

In [ ]:
# Consistencia entre la bitácora y las carpetas de fotos
print("IDs de la bitácora sin carpeta de fotos:", ordenar_ids(ids_bit - ids_fotos))
print("Carpetas de fotos sin registro en la bitácora:", ordenar_ids(ids_fotos - ids_bit))

filas_por_id = bit["id_vaca"].value_counts()
carpetas_por_id = fotos[["id_vaca", "descarga"]].drop_duplicates()["id_vaca"].value_counts()
todos = filas_por_id.index.union(carpetas_por_id.index)
consistencia = pd.DataFrame({"filas en la bitácora": filas_por_id.reindex(todos, fill_value=0),
                             "carpetas de fotos": carpetas_por_id.reindex(todos, fill_value=0)})
distintas = consistencia[consistencia.iloc[:, 0] != consistencia.iloc[:, 1]]
print(f"IDs donde no coinciden filas y carpetas: {len(distintas)}")
display(distintas)

# Hora y BCS en la bitácora
print(f"Horas con 'x': {(formato_hora == 'x (sin dato)').sum()} | con otro formato: {(formato_hora == 'otro formato').sum()}")
display(bit.loc[formato_hora == "otro formato", ["fila_excel", "id_vaca", "hora_original"]])
fuera = bit[bit["bcs"].isna() | (bit["bcs"] < 1) | (bit["bcs"] > 5)]
print(f"BCS vacíos o fuera de 1 a 5: {len(fuera)}")

In [ ]:
# Orientación: los JPG con orientación EXIF distinta de 1 se ven girados si el programa no la aplica
# (en los HEIC, la rotación va dentro del formato y la librería la aplica al abrirlos)
jpg = unicas[unicas["ext"].isin([".jpg", ".jpeg"])]
giradas = jpg[jpg["orientacion"].fillna(1).astype(int) != 1]
print(f"Fotos JPG: {len(jpg)} | con orientación EXIF distinta de 1: {len(giradas)}")
display(giradas.groupby(giradas["camara"].replace("", "(sin dato)")).size().rename_axis("cámara").to_frame("fotos giradas"))

### Resumen de calidad

In [ ]:
nombres_compartidas = "; ".join(f"{k} ({v})" for k, v in grupos_compartidos.items()) or "ninguna"
con_distinto = ", ".join(k for k, v in dobles.items() if v == "contenido distinto") or "ninguna"
calidad = pd.DataFrame([
    ["Archivos repetidos", f"{copias} copias; {len(dobles)} vacas con más de una carpeta; {numeradas} nombres tipo '(1)'",
     "3.2 Limpiar: dejar una sola copia por contenido"],
    ["Fotos en más de una vaca", f"{len(compartidas)} fotos idénticas: {nombres_compartidas}",
     "3.2 Limpiar: quitarlas, porque no se sabe a qué vaca pertenecen"],
    ["Carpetas con contenido distinto para una vaca", con_distinto,
     "3.2 Limpiar: no se puede saber qué fotos van con cada evaluación"],
    ["Formatos y resoluciones mezclados",
     f"{', '.join(f'{k} ({v})' for k, v in unicas['ext'].value_counts().items())}; {resolucion.nunique()} resoluciones",
     "3.5 Formatear: un solo formato antes de entrenar"],
    ["Fotos giradas", f"{len(giradas)} JPG con orientación EXIF distinta de 1",
     "3.5 Formatear: enderezarlas al convertir"],
    ["Hora en la bitácora", f"{(formato_hora == 'x (sin dato)').sum()} con 'x' y {(formato_hora == 'otro formato').sum()} con otro formato",
     "No se usa en el modelo; se conserva como registro"],
    ["Vacas con más de una evaluación", f"{len(pares)} vacas; diferencia media {dif.mean():.3f} y máxima {dif.max():.2f}",
     "3.3 Construir: definir la etiqueta de entrenamiento"],
    ["Vista de cada foto", "No viene en los nombres ni en los metadatos",
     "3.3 Construir: marcar la vista de cada foto"],
], columns=["aspecto", "evidencia", "tratamiento (fase 3)"])
with pd.option_context("display.max_colwidth", None):
    display(calidad)